In [40]:
!pip install gensim torch matplotlib seaborn scikit-learn pandas numpy tqdm

In [41]:
import json
import os
import random
import time
import copy
import re
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter
import zipfile
import tempfile
import shutil
from itertools import product

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import gensim.downloader as api
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from tqdm.auto import tqdm

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATASET = "IMDB_512"
MODEL = "LSTM"
MODEL_NAME = "BiLSTM-GloVe-300d"
NUM_EPOCHS = 10
BATCH_SIZE = 64
MAX_LENGTH = 50000
EMBED_DIM = 300
MIN_FREQ = 2
CLIP_GRAD = 1.0
OPTIMIZER_NAME = "Adam"
BIDIRECTIONAL = True
TEST_LABELS = True
TEXT_COLUMN = "text_original"
LABEL_COLUMN = "label"
ROOT_DIR = "/content/drive/MyDrive/NLP Sentiments/NLP_FAC"

# Parameters from metrics.json (MAX_LENGTH overridden to 512)
HIDDEN_SIZE = 128
NUM_LAYERS = 2
LEARNING_RATE = 0.001
DROPOUT = 0.5

print(f"Dataset:       {DATASET}")
print(f"Model:         {MODEL_NAME}")
print(f"Epochs:        {NUM_EPOCHS}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Max length:    {MAX_LENGTH}")
print(f"Hidden size:   {HIDDEN_SIZE}")
print(f"Num layers:    {NUM_LAYERS}")
print(f"LR:            {LEARNING_RATE}")
print(f"Dropout:       {DROPOUT}")
print(f"Embed dim:     {EMBED_DIM}")
print(f"Bidirectional: {BIDIRECTIONAL}")
print(f"Optimizer:     {OPTIMIZER_NAME}")
print(f"Seed:          {SEED}")


Device: cuda
Dataset:       IMDB_512
Model:         BiLSTM-GloVe-300d
Epochs:        10
Batch size:    64
Max length:    50000
Hidden size:   128
Num layers:    2
LR:            0.001
Dropout:       0.5
Embed dim:     300
Bidirectional: True
Optimizer:     Adam
Seed:          42


In [42]:
dataset_path = "/content/drive/MyDrive/NLP Sentiments/NLP_FAC/Datasets/IMDB"

def load_split(split_name):
    file_path = os.path.join(dataset_path, f"{split_name}.json")
    if not os.path.exists(file_path):
        print(f"  {split_name}.json not found in {dataset_path}")
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  {split_name}.json: {len(df)} samples")
    return df

print(f"Loading data from: {dataset_path}\n")
df_train = load_split("train")
df_validation = load_split("validation")
df_test = load_split("test")

if df_train is not None:
    print(f"\nFirst 3 train examples:")
    display(df_train.head(3))


Loading data from: /content/drive/MyDrive/NLP Sentiments/NLP_FAC/Datasets/IMDB

  train.json: 20000 samples
  validation.json: 5000 samples
  test.json: 25000 samples

First 3 train examples:


,text_original,text_preprocessed,label
0,I have always been a huge James Bond fanatic! ...,i have always been a huge james bond fanatic! ...,1
1,I am a Christian and I say this movie had terr...,i am a christian and i say this movie had terr...,0
2,"Neatly sandwiched between THE STRANGER, a smal...","neatly sandwiched between the stranger, a smal...",1


In [43]:
def simple_tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

train_texts = df_train[TEXT_COLUMN].fillna("").tolist()
train_labels = df_train[LABEL_COLUMN].tolist()
val_texts = df_validation[TEXT_COLUMN].fillna("").tolist()
val_labels = df_validation[LABEL_COLUMN].tolist()
test_texts = df_test[TEXT_COLUMN].fillna("").tolist()

y_test = None
if TEST_LABELS and LABEL_COLUMN in df_test.columns:
    y_test = df_test[LABEL_COLUMN].tolist()

# Build vocabulary from train set
print("Building vocabulary from train set...")
word_counts = Counter()
for text in train_texts:
    word_counts.update(simple_tokenize(text))

PAD_IDX = 0
UNK_IDX = 1
vocab_words = [w for w, c in word_counts.items() if c >= MIN_FREQ]
word2idx = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
for i, w in enumerate(vocab_words):
    word2idx[w] = i + 2
vocab_size = len(word2idx)
print(f"Vocabulary: {vocab_size} words (min_freq={MIN_FREQ})")

# Load GloVe embeddings
print(f"\nLoading GloVe embeddings ({EMBED_DIM}d)...")
glove_model = api.load(f"glove-wiki-gigaword-{EMBED_DIM}")
print(f"GloVe loaded: {len(glove_model)} words, {glove_model.vector_size}d")

# Build embedding matrix
print("Building embedding matrix...")
np.random.seed(SEED)
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
found = 0
for word, idx in word2idx.items():
    if word in glove_model:
        embedding_matrix[idx] = glove_model[word]
        found += 1
    elif idx == UNK_IDX:
        embedding_matrix[idx] = np.random.normal(0, 0.1, EMBED_DIM)
print(f"GloVe coverage: {found}/{vocab_size} ({found/vocab_size*100:.1f}%)")

# Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = simple_tokenize(self.texts[idx])
        indices = [word2idx.get(t, UNK_IDX) for t in tokens][:MAX_LENGTH]
        length = max(len(indices), 1)
        if len(indices) == 0:
            indices = [UNK_IDX]
        indices = indices + [PAD_IDX] * (MAX_LENGTH - len(indices))
        item = {
            "input_ids": torch.tensor(indices, dtype=torch.long),
            "lengths": torch.tensor(length, dtype=torch.long),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TextDataset(train_texts, train_labels)
val_dataset = TextDataset(val_texts, val_labels)
test_dataset = TextDataset(test_texts, y_test)

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

Building vocabulary from train set...
Vocabulary: 42920 words (min_freq=2)

Loading GloVe embeddings (300d)...
GloVe loaded: 400000 words, 300d
Building embedding matrix...
GloVe coverage: 40636/42920 (94.7%)

Train samples: 20000
Val samples:   5000
Test samples:  25000


In [44]:
num_labels = len(set(train_labels))

class LSTMClassifier(nn.Module):
    def __init__(self, emb_matrix, hidden_size, num_layers, num_classes, dropout, bidirectional=True):
        super().__init__()
        vs, ed = emb_matrix.shape
        self.embedding = nn.Embedding(vs, ed, padding_idx=0)
        self.embedding.weight = nn.Parameter(torch.tensor(emb_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(
            ed, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        factor = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_size * factor, num_classes)
        self.bidirectional = bidirectional

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        hidden = self.dropout(hidden)
        return self.fc(hidden)

def create_model(hidden_size, num_layers, dropout):
    m = LSTMClassifier(embedding_matrix, hidden_size, num_layers, num_labels, dropout, BIDIRECTIONAL)
    m.to(device)
    return m

def build_optimizer(name, model_params, lr):
    optimizers = {
        "Adam": lambda: torch.optim.Adam(model_params, lr=lr),
        "AdamW": lambda: torch.optim.AdamW(model_params, lr=lr),
        "SGD": lambda: torch.optim.SGD(model_params, lr=lr, momentum=0.9),
    }
    if name not in optimizers:
        raise ValueError(f"Unknown optimizer: {name}. Choose from: {list(optimizers.keys())}")
    return optimizers[name]()

criterion = nn.CrossEntropyLoss()

print(f"Num labels: {num_labels}")
print("LSTMClassifier, create_model(), build_optimizer() ready.")

Num labels: 2
LSTMClassifier, create_model(), build_optimizer() ready.


In [45]:
# ============================================================
# Training with specified parameters
# ============================================================

set_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = create_model(HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
optimizer = build_optimizer(OPTIMIZER_NAME, model.parameters(), LEARNING_RATE)

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = -1
best_val_loss = float('inf')
best_epoch = 0
best_model_state = None

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ---------- Train ----------
    model.train()
    epoch_loss = 0
    epoch_preds, epoch_labels = [], []

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]")
    for batch in train_bar:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"]
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_GRAD)
        optimizer.step()

        epoch_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        epoch_preds.extend(preds.cpu().numpy())
        epoch_labels.extend(labels.cpu().numpy())
        train_bar.set_postfix(loss=loss.item())

    train_loss = epoch_loss / len(train_loader)
    train_acc = accuracy_score(epoch_labels, epoch_preds)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # ---------- Validation ----------
    model.eval()
    val_loss = 0
    val_preds, val_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"]
            labels = batch["labels"].to(device)

            outputs = model(input_ids, lengths)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_labels_list, val_preds)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

train_time = time.time() - start_time
entries_per_second = (len(train_labels) * NUM_EPOCHS) / train_time

print(f"Training done in {train_time:.1f}s | Best Epoch: {best_epoch} | Best Val Acc: {best_val_acc:.4f}")
model.load_state_dict(best_model_state)


Epoch 1/10 [Train]:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 0.5264 | Val Loss: 0.4151 | Train Acc: 0.7374 | Val Acc: 0.8214


Epoch 2/10 [Train]:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 0.4039 | Val Loss: 0.3786 | Train Acc: 0.8256 | Val Acc: 0.8342


Epoch 3/10 [Train]:   0%|          | 0/313 [00:00<?, ?it/s]

...

Epoch 9/10 [Train]:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 0.1488 | Val Loss: 0.3969 | Train Acc: 0.9443 | Val Acc: 0.8802


Epoch 10/10 [Train]:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.1199 | Val Loss: 0.4221 | Train Acc: 0.9561 | Val Acc: 0.8774
Training done in 1772.1s | Best Epoch: 7 | Best Val Acc: 0.8938


<All keys matched successfully>

In [46]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
optimizer_info = {'name': OPTIMIZER_NAME, 'lr': LEARNING_RATE}
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')


Model params: 13,712,098


In [47]:
model.eval()

def predict(loader):
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"]
            has_labels = "labels" in batch and batch["labels"] is not None
            if has_labels:
                labels = batch["labels"].to(device)
                outputs = model(input_ids, lengths)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
                all_labels.extend(labels.cpu().numpy())
            else:
                outputs = model(input_ids, lengths)
            probs = torch.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    avg_loss = total_loss / len(loader) if all_labels else None
    return np.array(all_preds), np.array(all_labels) if all_labels else None, avg_loss, np.array(all_probs)

y_train_pred, y_train_true, train_loss_final, train_probs = predict(train_loader)
y_val_pred, y_val_true, val_loss_final, val_probs = predict(val_loader)
y_test_pred, y_test_true, test_loss_final, test_probs = predict(test_loader)

train_accuracy = accuracy_score(y_train_true, y_train_pred)

val_accuracy = accuracy_score(y_val_true, y_val_pred)
val_precision_macro = precision_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_recall_macro = recall_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_f1_macro = f1_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_precision_per_class = precision_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_recall_per_class = recall_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_f1_per_class = f1_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_conf_matrix = confusion_matrix(y_val_true, y_val_pred).tolist()

val_fpr, val_tpr, _ = roc_curve(y_val_true, val_probs[:, 1])
val_auc = auc(val_fpr, val_tpr)
val_pr_precision, val_pr_recall, _ = precision_recall_curve(y_val_true, val_probs[:, 1])
val_avg_precision = average_precision_score(y_val_true, val_probs[:, 1])

class_labels = sorted(np.unique(y_val_true).tolist())

if TEST_LABELS and y_test_true is not None:
    test_accuracy = accuracy_score(y_test_true, y_test_pred)
    test_precision_macro = precision_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_recall_macro = recall_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_precision_per_class = precision_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_recall_per_class = recall_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_f1_per_class = f1_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_conf_matrix = confusion_matrix(y_test_true, y_test_pred).tolist()

    test_fpr, test_tpr, _ = roc_curve(y_test_true, test_probs[:, 1])
    test_auc = auc(test_fpr, test_tpr)
    test_pr_precision, test_pr_recall, _ = precision_recall_curve(y_test_true, test_probs[:, 1])
    test_avg_precision = average_precision_score(y_test_true, test_probs[:, 1])

print("=" * 60)
print(f"  RESULTS {MODEL} on {DATASET} (best epoch: {best_epoch})")
print("=" * 60)
print(f"  Train Accuracy:      {train_accuracy:.4f}   | Loss: {train_loss_final:.4f}")
print(f"  Val Accuracy:        {val_accuracy:.4f}   | Loss: {val_loss_final:.4f}")
print(f"  Val AUC:             {val_auc:.4f}")

if not TEST_LABELS:
    unique, counts = np.unique(y_test_pred, return_counts=True)
    print(f"  Test prediction distribution: {dict(zip(unique.tolist(), counts.tolist()))}")

print(f"\n  Classification Report (VALIDATION):")
print(classification_report(y_val_true, y_val_pred, zero_division=0))

  RESULTS LSTM on IMDB_512 (best epoch: 7)
  Train Accuracy:      0.9423   | Loss: 0.1566
  Val Accuracy:        0.8938   | Loss: 0.3096
  Val AUC:             0.9545

  Classification Report (VALIDATION):
              precision    recall  f1-score   support

           0       0.89      0.90      0.89      2500
           1       0.90      0.89      0.89      2500

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000



In [48]:
if TEST_LABELS and y_test_true is not None:
    cm_array = np.array(test_conf_matrix)
    cm_labels = sorted(np.unique(y_test_true).tolist())
else:
    cm_array = np.array(val_conf_matrix)
    cm_labels = sorted(np.unique(y_val_true).tolist())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=cm_labels, yticklabels=cm_labels, ax=ax_cm)
ax_cm.set_xlabel("Predicted")
ax_cm.set_ylabel("Actual")
ax_cm.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
fig_cm.tight_layout()
plt.show()

print(f"Classes: {cm_labels}")
print(f"Matrix:\n{cm_array}")

Classes: [0, 1]
Matrix:
[[11282  1218]
 [ 1331 11169]]


In [49]:
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

if TEST_LABELS and y_test_true is not None:
    ax_roc.plot(test_fpr, test_tpr, linewidth=2, color="#1f77b4",
                label=f"Test ROC (AUC = {test_auc:.4f})")
else:
    ax_roc.plot(val_fpr, val_tpr, linewidth=2, color="#1f77b4",
                label=f"Validation ROC (AUC = {val_auc:.4f})")

ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_roc.legend(loc="lower right", fontsize=12)
ax_roc.grid(True, alpha=0.3)
fig_roc.tight_layout()
plt.show()

In [50]:
fig_pr, ax_pr = plt.subplots(figsize=(8, 6))

if TEST_LABELS and y_test_true is not None:
    ax_pr.plot(test_pr_recall, test_pr_precision, linewidth=2, color="#1f77b4",
               label=f"Test PR (AP = {test_avg_precision:.4f})")
else:
    ax_pr.plot(val_pr_recall, val_pr_precision, linewidth=2, color="#1f77b4",
               label=f"Validation PR (AP = {val_avg_precision:.4f})")

ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_pr.legend(loc="lower left", fontsize=12)
ax_pr.grid(True, alpha=0.3)
ax_pr.set_xlim([0, 1])
ax_pr.set_ylim([0, 1.05])
fig_pr.tight_layout()
plt.show()

In [51]:
if TEST_LABELS:
    pred_source = y_test_pred
    source_label = "Test"
else:
    pred_source = y_val_pred
    source_label = "Validation"

counts = [np.sum(pred_source == 0), np.sum(pred_source == 1)]
labels_bar = ["Negative (0)", "Positive (1)"]
colors = ["#e74c3c", "#2ecc71"]

fig_dist, ax_dist = plt.subplots(figsize=(6, 5))
bars = ax_dist.bar(labels_bar, counts, color=colors, edgecolor="black", width=0.5)
for bar, count in zip(bars, counts):
    ax_dist.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                 str(count), ha="center", va="bottom", fontweight="bold", fontsize=12)
ax_dist.set_xlabel("Predicted Sentiment")
ax_dist.set_ylabel("Number of Samples")
ax_dist.set_title(f"{MODEL} - {DATASET}", fontweight="bold")
fig_dist.tight_layout()
plt.show()

print(f"Negative: {counts[0]}, Positive: {counts[1]}")

Negative: 12613, Positive: 12387


In [52]:
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig_acc, ax_acc = plt.subplots(figsize=(9, 5))
ax_acc.plot(epochs_range, train_accs, label="Train Accuracy", linewidth=2, color="#1f77b4")
ax_acc.plot(epochs_range, val_accs, label="Validation Accuracy", linewidth=2, color="#ff7f0e")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy")
ax_acc.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_acc.legend(loc="lower right")
ax_acc.grid(True, alpha=0.3)
fig_acc.tight_layout()
plt.show()

print(f"Last epoch -- Train: {train_accs[-1]:.4f}, Val: {val_accs[-1]:.4f}")

Last epoch -- Train: 0.9561, Val: 0.8774


In [53]:
fig_loss, ax_loss = plt.subplots(figsize=(9, 5))
ax_loss.plot(epochs_range, train_losses, label="Train Loss", linewidth=2, color="#1f77b4")
ax_loss.plot(epochs_range, val_losses, label="Validation Loss", linewidth=2, color="#ff7f0e")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss (CrossEntropy)")
ax_loss.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_loss.legend(loc="upper right")
ax_loss.grid(True, alpha=0.3)
fig_loss.tight_layout()
plt.show()

print(f"Last epoch -- Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")

Last epoch -- Train Loss: 0.1199, Val Loss: 0.4221


In [54]:
def save_results(model_obj, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, num_epochs, best_epoch, batch_size, learning_rate,
                 hidden_size, num_layers, dropout, embed_dim, max_length, vocab_size, bidirectional,
                 test_labels, text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir,
                 optimizer_info=None):

    model_size_bytes = sum(p.numel() * p.element_size() for p in model_obj.parameters())
    model_size_mb = model_size_bytes / (1024 * 1024)

    timestamp_str = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    run_name = f"BiLSTM_h{hidden_size}_l{num_layers}_lr{learning_rate}_{timestamp_str}"

    output_dir = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(output_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(output_dir, fig_name), dpi=150, bbox_inches="tight")
    print(f"Charts saved: {list(figures.keys())}")

    total_params = sum(p.numel() for p in model_obj.parameters())
    trainable_params = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

    metrics = {
        "model": model_name,
        "model_name": "BiLSTM-GloVe-300d",
        "dataset": dataset,
        "run_name": run_name,
        "timestamp": timestamp_str,
        "seed": SEED,
        "model_params": {
            "num_epochs": num_epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "embed_dim": embed_dim,
            "max_length": max_length,
            "vocab_size": vocab_size,
            "bidirectional": bidirectional,
            "total_params": total_params,
            "trainable_params": trainable_params,
        },
        "model_size": {
            "bytes": model_size_bytes,
            "MB": round(model_size_mb, 4)
        },
        "training": {
            "training_time_seconds": round(train_time, 4),
            "entries_per_second": round(entries_per_second, 2),
            "num_epochs": num_epochs,
            "best_epoch": best_epoch,
            "train_samples": len(df_train),
        },
        "data": {
            "train_samples": len(df_train),
            "validation_samples": len(df_validation),
            "test_samples": len(df_test) if df_test is not None else 0,
            "text_column": text_column,
            "label_column": label_column,
            "classes": class_labels,
        },
        "optimizer": optimizer_info or {},
        "train_results": train_results,
        "validation_results": val_results,
    }

    if test_labels and test_results is not None:
        metrics["test_results"] = test_results

    metrics_path = os.path.join(output_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"metrics.json saved in: {output_dir}")
    return output_dir, metrics_path, run_name


figures = {
    "confusion_matrix.png": fig_cm,
    "roc_curve.png": fig_roc,
    "pr_curve.png": fig_pr,
    "prediction_distribution.png": fig_dist,
    "accuracy_curves.png": fig_acc,
    "loss_curves.png": fig_loss,
}

train_res = {
    "accuracy": round(train_accuracy, 4),
    "loss": round(train_loss_final, 4),
}

val_res = {
    "accuracy": round(val_accuracy, 4),
    "loss": round(val_loss_final, 4),
    "precision_macro": round(val_precision_macro, 4),
    "recall_macro": round(val_recall_macro, 4),
    "f1_macro": round(val_f1_macro, 4),
    "auc": round(val_auc, 4),
    "avg_precision": round(val_avg_precision, 4),
    "precision_per_class": [round(p, 4) for p in val_precision_per_class],
    "recall_per_class": [round(r, 4) for r in val_recall_per_class],
    "f1_per_class": [round(f, 4) for f in val_f1_per_class],
    "confusion_matrix": val_conf_matrix,
}

test_res = None
if TEST_LABELS and y_test_true is not None:
    test_res = {
        "accuracy": round(test_accuracy, 4),
        "loss": round(test_loss_final, 4),
        "precision_macro": round(test_precision_macro, 4),
        "recall_macro": round(test_recall_macro, 4),
        "f1_macro": round(test_f1_macro, 4),
        "auc": round(test_auc, 4),
        "avg_precision": round(test_avg_precision, 4),
        "precision_per_class": [round(p, 4) for p in test_precision_per_class],
        "recall_per_class": [round(r, 4) for r in test_recall_per_class],
        "f1_per_class": [round(f, 4) for f in test_f1_per_class],
        "confusion_matrix": test_conf_matrix,
    }

output_dir, metrics_path, run_name = save_results(
    model_obj=model, figures=figures,
    train_results=train_res, val_results=val_res, test_results=test_res,
    train_time=train_time, entries_per_second=entries_per_second,
    dataset=DATASET, model_name=MODEL,
    num_epochs=NUM_EPOCHS, best_epoch=best_epoch, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT,
    embed_dim=EMBED_DIM, max_length=MAX_LENGTH, vocab_size=vocab_size,
    bidirectional=BIDIRECTIONAL,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,
    optimizer_info=optimizer_info,
)

Charts saved: ['confusion_matrix.png', 'roc_curve.png', 'pr_curve.png', 'prediction_distribution.png', 'accuracy_curves.png', 'loss_curves.png']
metrics.json saved in: /content/drive/MyDrive/NLP Sentiments/NLP_FAC/LSTM/IMDB_512/BiLSTM_h128_l2_lr0.001_14_04_2026_15_07_15


In [55]:
with open(metrics_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"{'='*60}")
print(f"  RUN SUMMARY: {run_name}")
print(f"{'='*60}")
print(f"  Model:              {saved['model']}")
print(f"  Model name:         {saved['model_name']}")
print(f"  Dataset:            {saved['dataset']}")
print(f"  Training time:      {saved['training']['training_time_seconds']}s")
print(f"  Entries/sec:        {saved['training']['entries_per_second']}")
print(f"  Best epoch:         {saved['training']['best_epoch']}/{saved['training']['num_epochs']}")
print(f"  Model size:         {saved['model_size']['MB']} MB")
print(f"  Total params:       {saved['model_params']['total_params']:,}")
print(f"  Train Accuracy:     {saved['train_results']['accuracy']}")
print(f"  Train Loss:         {saved['train_results']['loss']}")
print(f"  Val Accuracy:       {saved['validation_results']['accuracy']}")
print(f"  Val Loss:           {saved['validation_results']['loss']}")
print(f"  Val F1 (macro):     {saved['validation_results']['f1_macro']}")
if "test_results" in saved:
    print(f"  Test:               saved in metrics.json")
else:
    print(f"  Test:               N/A (TEST_LABELS=False)")
print(f"{'='*60}")
print(f"\n  Files saved in: {output_dir}")
for f_name in os.listdir(output_dir):
    f_size = os.path.getsize(os.path.join(output_dir, f_name))
    print(f"    {f_name} ({f_size:,} bytes)")

  RUN SUMMARY: BiLSTM_h128_l2_lr0.001_14_04_2026_15_07_15
  Model:              LSTM
  Model name:         BiLSTM-GloVe-300d
  Dataset:            IMDB_512
  Training time:      1772.1113s
  Entries/sec:        112.86
  Best epoch:         7/10
  Model size:         52.3075 MB
  Total params:       13,712,098
  Train Accuracy:     0.9423
  Train Loss:         0.1566
  Val Accuracy:       0.8938
  Val Loss:           0.3096
  Val F1 (macro):     0.8938
  Test:               saved in metrics.json

  Files saved in: /content/drive/MyDrive/NLP Sentiments/NLP_FAC/LSTM/IMDB_512/BiLSTM_h128_l2_lr0.001_14_04_2026_15_07_15
    confusion_matrix.png (28,563 bytes)
    roc_curve.png (65,432 bytes)
    pr_curve.png (41,391 bytes)
    prediction_distribution.png (37,344 bytes)
    accuracy_curves.png (55,160 bytes)
    loss_curves.png (68,452 bytes)
    metrics.json (2,081 bytes)


In [56]:
print(json.dumps(saved, indent=2, ensure_ascii=False))

{
  "model": "LSTM",
  "model_name": "BiLSTM-GloVe-300d",
  "dataset": "IMDB_512",
  "run_name": "BiLSTM_h128_l2_lr0.001_14_04_2026_15_07_15",
  "timestamp": "14_04_2026_15_07_15",
  "seed": 42,
  "model_params": {
    "num_epochs": 10,
    "batch_size": 64,
    "learning_rate": 0.001,
    "hidden_size": 128,
    "num_layers": 2,
    "dropout": 0.5,
    "embed_dim": 300,
    "max_length": 50000,
    "vocab_size": 42920,
    "bidirectional": true,
    "total_params": 13712098,
    "trainable_params": 836098
  },
  "model_size": {
    "bytes": 54848392,
    "MB": 52.3075
  },
  "training": {
    "training_time_seconds": 1772.1113,
    "entries_per_second": 112.86,
    "num_epochs": 10,
    "best_epoch": 7,
    "train_samples": 20000
  },
  "data": {
    "train_samples": 20000,
    "validation_samples": 5000,
    "test_samples": 25000,
    "text_column": "text_original",
    "label_column": "label",
    "classes": [
      0,
      1
    ]
  },
  "optimizer": {
    "name": "Adam",
    "lr"